In [1]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub safetensors datasets

import torch, transformers, sys
print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, "| VRAM GB:", round(p.total_memory / 1e9, 1))

import shutil
print("Free disk GB:", round(shutil.disk_usage("/").free / 1e9, 1))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
Python: 3.13.16
torch: 2.14.1+cu130
transformers: 5.24.0
CUDA available: True
GPU: Tesla T4 | VRAM GB: 15.6
Free disk GB: 120.3


In [2]:
import json
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()

def fetch_model_report(model_id: str) -> dict:
    """Inspect a HF model without downloading weights."""
    info = api.model_info(model_id, files_metadata=True)

    # --- files ---
    files = {s.rfilename: (s.size or 0) for s in info.siblings}
    safetensors = {n: s for n, s in files.items() if n.endswith(".safetensors")}
    pickles     = {n: s for n, s in files.items() if n.endswith((".bin", ".pt", ".pth", ".pkl"))}
    ggufs       = {n: s for n, s in files.items() if n.endswith(".gguf")}

    # --- config (tiny JSON, safe to download) ---
    cfg_path = hf_hub_download(model_id, "config.json")
    with open(cfg_path) as f:
        cfg = json.load(f)

    # --- parameter count: prefer the hub's safetensors metadata ---
    params = None
    if getattr(info, "safetensors", None) and info.safetensors.total:
        params = info.safetensors.total
    elif safetensors:
        params = int(sum(safetensors.values()) / 2)  # rough guess: fp16

    # --- safety flags ---
    flags = []
    if cfg.get("auto_map"):
        flags.append("REMOTE_CODE: config uses auto_map (needs trust_remote_code)")
    if pickles and not safetensors:
        flags.append("PICKLE_ONLY: weights are pickle files (unsafe to load)")
    if getattr(info, "gated", False):
        flags.append("GATED: requires accepting license / HF token")
    if cfg.get("quantization_config"):
        flags.append("ALREADY_QUANTIZED: model already has a quantization_config")

    fp16_gb = round(params * 2 / 1e9, 2) if params else None
    q4_gb   = round(params * 0.5 / 1e9, 2) if params else None

    return {
        "model_id": model_id,
        "architecture": (cfg.get("architectures") or ["?"])[0],
        "model_type": cfg.get("model_type"),
        "params_millions": round(params / 1e6, 1) if params else None,
        "layers": cfg.get("num_hidden_layers"),
        "hidden_size": cfg.get("hidden_size"),
        "vocab_size": cfg.get("vocab_size"),
        "native_dtype": cfg.get("dtype") or cfg.get("torch_dtype"),
        "safetensors_files": len(safetensors),
        "total_weight_gb": round(sum(safetensors.values()) / 1e9, 2),
        "est_fp16_gb": fp16_gb,
        "est_4bit_gb": q4_gb,
        "flags": flags,
        "safe_to_proceed": not any(x.startswith(("REMOTE_CODE", "PICKLE_ONLY", "GATED")) for x in flags),
    }

# Try it on two small models
for mid in ["Qwen/Qwen2.5-0.5B", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"]:
    print(json.dumps(fetch_model_report(mid), indent=2))
    print("-" * 60)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

{
  "model_id": "Qwen/Qwen2.5-0.5B",
  "architecture": "Qwen2ForCausalLM",
  "model_type": "qwen2",
  "params_millions": 494.0,
  "layers": 24,
  "hidden_size": 896,
  "vocab_size": 151936,
  "native_dtype": "bfloat16",
  "safetensors_files": 1,
  "total_weight_gb": 0.99,
  "est_fp16_gb": 0.99,
  "est_4bit_gb": 0.25,
  "flags": [],
  "safe_to_proceed": true
}
------------------------------------------------------------


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

{
  "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
  "architecture": "LlamaForCausalLM",
  "model_type": "llama",
  "params_millions": 1100.0,
  "layers": 22,
  "hidden_size": 2048,
  "vocab_size": 32000,
  "native_dtype": "bfloat16",
  "safetensors_files": 1,
  "total_weight_gb": 2.2,
  "est_fp16_gb": 2.2,
  "est_4bit_gb": 0.55,
  "flags": [],
  "safe_to_proceed": true
}
------------------------------------------------------------


In [3]:
import time, os, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def gpu_gb():
    return round(torch.cuda.memory_allocated() / 1e9, 3)

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def dir_size_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return round(total / 1e9, 3)

def quantize_model(model_id: str, out_dir: str = "/content/q4_model") -> dict:
    tok = AutoTokenizer.from_pretrained(model_id)

    # ---- baseline fp16 ----
    free_gpu()
    base = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="cuda"
    )
    base_mem = gpu_gb()
    inputs = tok("The capital of France is", return_tensors="pt").to("cuda")
    t = time.time()
    out = base.generate(**inputs, max_new_tokens=30, do_sample=False)
    base_time = round(time.time() - t, 2)
    base_text = tok.decode(out[0], skip_special_tokens=True)
    del base
    free_gpu()

    # ---- 4-bit NF4 ----
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    t = time.time()
    qm = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map="cuda"
    )
    load_time = round(time.time() - t, 2)
    q_mem = gpu_gb()
    t = time.time()
    out = qm.generate(**inputs, max_new_tokens=30, do_sample=False)
    q_time = round(time.time() - t, 2)
    q_text = tok.decode(out[0], skip_special_tokens=True)

    qm.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)

    return {
        "model_id": model_id,
        "fp16_gpu_gb": base_mem,
        "q4_gpu_gb": q_mem,
        "compression_x": round(base_mem / q_mem, 2),
        "q4_disk_gb": dir_size_gb(out_dir),
        "fp16_gen_sec": base_time,
        "q4_gen_sec": q_time,
        "q4_load_sec": load_time,
        "fp16_output": base_text,
        "q4_output": q_text,
        "saved_to": out_dir,
    }

result = quantize_model("Qwen/Qwen2.5-0.5B")
for k, v in result.items():
    print(f"{k}: {v}")

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model_id: Qwen/Qwen2.5-0.5B
fp16_gpu_gb: 0.988
q4_gpu_gb: 0.466
compression_x: 2.12
q4_disk_gb: 0.469
fp16_gen_sec: 5.32
q4_gen_sec: 2.71
q4_load_sec: 17.3
fp16_output: The capital of France is Paris. It is the largest city in Europe and the second largest in the world. It is also the capital of France, the second largest country in
q4_output: The capital of France is Paris. It is the capital of France, the largest city in Europe, and the second largest city in the United Kingdom. It is the largest city
saved_to: /content/q4_model


In [4]:
import math, gc, torch, json, hashlib, os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

# load dataset once, using the full namespaced repo id
_wt = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
WIKI_TEXT = "\n\n".join(_wt["text"])

def eval_perplexity(model, tok, max_tokens=40000, seq_len=512):
    ids = tok(WIKI_TEXT, return_tensors="pt").input_ids[:, :max_tokens]
    nll, count = 0.0, 0
    model.eval()
    with torch.no_grad():
        for i in range(0, ids.size(1) - 1, seq_len):
            chunk = ids[:, i:i + seq_len].to("cuda")
            if chunk.size(1) < 2:
                break
            out = model(chunk, labels=chunk)
            n = chunk.size(1) - 1
            nll += out.loss.item() * n
            count += n
    return math.exp(nll / count)

def folder_sha256(path):
    h = hashlib.sha256()
    for f in sorted(os.listdir(path)):
        if f.endswith(".safetensors"):
            with open(os.path.join(path, f), "rb") as fh:
                for block in iter(lambda: fh.read(1 << 20), b""):
                    h.update(block)
    return h.hexdigest()

def validate(model_id, q_dir="/content/q4_model", max_ppl_increase_pct=10.0):
    tok = AutoTokenizer.from_pretrained(model_id)

    base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda")
    base_ppl = eval_perplexity(base, tok)
    del base; free_gpu()

    qm = AutoModelForCausalLM.from_pretrained(q_dir, device_map="cuda")
    q_ppl = eval_perplexity(qm, tok)
    del qm; free_gpu()

    delta_pct = round((q_ppl - base_ppl) / base_ppl * 100, 2)
    return {
        "model_id": model_id,
        "fp16_perplexity": round(base_ppl, 3),
        "q4_perplexity": round(q_ppl, 3),
        "ppl_increase_pct": delta_pct,
        "budget_pct": max_ppl_increase_pct,
        "verdict": "PASS" if delta_pct <= max_ppl_increase_pct else "FAIL",
        "artifact_sha256": folder_sha256(q_dir),
    }

report = validate("Qwen/Qwen2.5-0.5B")
print(json.dumps(report, indent=2))

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

... (4 outputs left out)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{
  "model_id": "Qwen/Qwen2.5-0.5B",
  "fp16_perplexity": 17.261,
  "q4_perplexity": 19.491,
  "ppl_increase_pct": 12.92,
  "budget_pct": 10.0,
  "verdict": "FAIL",
  "artifact_sha256": "b75f2948d8200490d3f632601518b576f0638ce22520ce2a38a507d60342533f"
}


In [5]:
import shutil, time, json, torch, os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def build_ladder():
    return [
        ("nf4+double_quant", BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)),
        ("nf4", BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=False, bnb_4bit_compute_dtype=torch.float16)),
        ("int8", BitsAndBytesConfig(load_in_8bit=True)),
    ]

def quantize_pipeline(model_id, max_ppl_increase_pct=10.0, out_dir="/content/out_model"):
    tok = AutoTokenizer.from_pretrained(model_id)

    # baseline once
    base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda")
    base_gb = round(base.get_memory_footprint() / 1e9, 3)
    base_ppl = eval_perplexity(base, tok)
    del base; free_gpu()

    attempts, winner = [], None
    for name, cfg in build_ladder():
        t = time.time()
        qm = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=cfg, device_map="cuda")
        q_gb = round(qm.get_memory_footprint() / 1e9, 3)
        q_ppl = eval_perplexity(qm, tok)
        delta = round((q_ppl - base_ppl) / base_ppl * 100, 2)
        ok = delta <= max_ppl_increase_pct
        attempts.append({
            "config": name, "size_gb": q_gb, "ppl": round(q_ppl, 3),
            "ppl_increase_pct": delta, "verdict": "PASS" if ok else "FAIL",
            "sec": round(time.time() - t, 1),
        })
        print(f"[{name}] size={q_gb} GB  ppl={q_ppl:.3f}  +{delta}%  -> {'PASS' if ok else 'FAIL'}")
        if ok:
            if os.path.exists(out_dir):
                shutil.rmtree(out_dir)
            qm.save_pretrained(out_dir); tok.save_pretrained(out_dir)
            winner = name
            del qm; free_gpu()
            break
        del qm; free_gpu()

    return {
        "model_id": model_id,
        "budget_pct": max_ppl_increase_pct,
        "fp16_size_gb": base_gb,
        "fp16_perplexity": round(base_ppl, 3),
        "attempts": attempts,
        "winner": winner,
        "artifact_dir": out_dir if winner else None,
        "artifact_sha256": folder_sha256(out_dir) if winner else None,
        "status": "SUCCESS" if winner else "NO_CONFIG_MET_BUDGET",
    }

final = quantize_pipeline("Qwen/Qwen2.5-0.5B", max_ppl_increase_pct=10.0)
print(json.dumps(final, indent=2))
import warnings, logging
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[nf4+double_quant] size=0.451 GB  ppl=19.491  +12.92%  -> FAIL


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[nf4] size=0.451 GB  ppl=19.496  +12.95%  -> FAIL


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.


[int8] size=0.63 GB  ppl=17.415  +0.89%  -> PASS


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "model_id": "Qwen/Qwen2.5-0.5B",
  "budget_pct": 10.0,
  "fp16_size_gb": 0.988,
  "fp16_perplexity": 17.261,
  "attempts": [
    {
      "config": "nf4+double_quant",
      "size_gb": 0.451,
      "ppl": 19.491,
      "ppl_increase_pct": 12.92,
      "verdict": "FAIL",
      "sec": 22.7
    },
    {
      "config": "nf4",
      "size_gb": 0.451,
      "ppl": 19.496,
      "ppl_increase_pct": 12.95,
      "verdict": "FAIL",
      "sec": 13.6
    },
    {
      "config": "int8",
      "size_gb": 0.63,
      "ppl": 17.415,
      "ppl_increase_pct": 0.89,
      "verdict": "PASS",
      "sec": 42.1
    }
  ],
  "winner": "int8",
  "artifact_dir": "/content/out_model",
  "artifact_sha256": "d77e8101b56bb22c8795327b521cc5e0e26976aadc15fe86a44a3c50f81fc857",
  "status": "SUCCESS"
}


In [6]:
!pip install -q crewai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [7]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

!ollama pull qwen2.5:7b
!ollama list

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

NAME          ID              SIZE      MODIFIED               
qwen2.5:7b    845dbda0ea48    4.7 GB    Less than a second ago    


In [10]:
import json, os, time, requests

class FetcherAgent:
    name = "Fetcher"
    def run(self, state):
        rep = fetch_model_report(state["model_id"])
        state["inspection"] = rep
        state["proceed"] = rep["safe_to_proceed"]
        state["log"].append(f"[Fetcher] {rep['architecture']}, {rep['params_millions']}M params, "
                            f"flags={rep['flags'] or 'none'} -> {'PROCEED' if state['proceed'] else 'STOP'}")
        return state

class QuantizerAgent:
    name = "Quantizer"
    def run(self, state):
        if not state["proceed"]:
            state["log"].append("[Quantizer] skipped (Fetcher said STOP)")
            return state
        rep = quantize_pipeline(state["model_id"], state["budget"])
        state["quant"] = rep
        state["log"].append(f"[Quantizer] status={rep['status']}, winner={rep['winner']}, "
                            f"attempts={len(rep['attempts'])}")
        return state

class ValidatorAgent:
    name = "Validator"
    def run(self, state):
        q = state.get("quant")
        if not q or q["status"] != "SUCCESS":
            state["verified"] = False
            state["log"].append("[Validator] nothing to verify")
            return state
        d = q["artifact_dir"]
        ok = os.path.exists(d) and folder_sha256(d) == q["artifact_sha256"]
        state["verified"] = ok
        state["final_size_gb"] = dir_size_gb(d)
        state["log"].append(f"[Validator] hash_match={ok}, disk={state['final_size_gb']} GB")
        return state

def llm_summary(facts: dict, model="qwen2.5:7b"):
    """Optional: ask local Ollama to write a short summary. Falls back to None."""
    try:
        r = requests.post("http://localhost:11434/api/generate", timeout=180, json={
            "model": model, "stream": False,
            "prompt": "Write a 3-sentence honest report for another AI agent from these facts. "
                      "Do not invent numbers.\n" + json.dumps(facts)})
        return r.json()["response"].strip()
    except Exception as e:
        return None

def quantize(model_id, max_ppl_increase_pct=10.0, use_llm_summary=False):
    state = {"model_id": model_id, "budget": max_ppl_increase_pct, "log": []}
    for agent in (FetcherAgent(), QuantizerAgent(), ValidatorAgent()):
        state = agent.run(state)
        print(state["log"][-1])

    q = state.get("quant") or {}
    winner = next((a for a in q.get("attempts", []) if a["config"] == q.get("winner")), None)
    report = {
        "model_id": model_id,
        "status": q.get("status", "STOPPED_BY_FETCHER"),
        "config": q.get("winner"),
        "fp16_size_gb": q.get("fp16_size_gb"),
        "final_size_gb": state.get("final_size_gb"),
        "ppl_increase_pct": winner["ppl_increase_pct"] if winner else None,
        "budget_pct": max_ppl_increase_pct,
        "hash_verified": state.get("verified"),
        "artifact_dir": q.get("artifact_dir"),
        "attempts": q.get("attempts", []),
    }
    if use_llm_summary:
        report["summary"] = llm_summary({k: v for k, v in report.items() if k != "attempts"})
    return report

final = quantize("Qwen/Qwen2.5-0.5B", max_ppl_increase_pct=10.0, use_llm_summary=False)
print(json.dumps(final, indent=2))

[Fetcher] Qwen2ForCausalLM, 494.0M params, flags=none -> PROCEED


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[nf4+double_quant] size=0.451 GB  ppl=19.491  +12.92%  -> FAIL


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[nf4] size=0.451 GB  ppl=19.496  +12.95%  -> FAIL


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[int8] size=0.63 GB  ppl=17.415  +0.89%  -> PASS


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[Quantizer] status=SUCCESS, winner=int8, attempts=3
[Validator] hash_match=True, disk=0.643 GB
{
  "model_id": "Qwen/Qwen2.5-0.5B",
  "status": "SUCCESS",
  "config": "int8",
  "fp16_size_gb": 0.988,
  "final_size_gb": 0.643,
  "ppl_increase_pct": 0.89,
  "budget_pct": 10.0,
  "hash_verified": true,
  "artifact_dir": "/content/out_model",
  "attempts": [
    {
      "config": "nf4+double_quant",
      "size_gb": 0.451,
      "ppl": 19.491,
      "ppl_increase_pct": 12.92,
      "verdict": "FAIL",
      "sec": 15.5
    },
    {
      "config": "nf4",
      "size_gb": 0.451,
      "ppl": 19.496,
      "ppl_increase_pct": 12.95,
      "verdict": "FAIL",
      "sec": 13.4
    },
    {
      "config": "int8",
      "size_gb": 0.63,
      "ppl": 17.415,
      "ppl_increase_pct": 0.89,
      "verdict": "PASS",
      "sec": 21.7
    }
  ]
}


In [11]:
import os, subprocess, json
print("functions alive:", all(n in globals() for n in
      ["fetch_model_report", "quantize_pipeline", "folder_sha256", "dir_size_gb", "quantize"]))
print("out_model exists:", os.path.exists("/content/out_model"))
if os.path.exists("/content/out_model"):
    print(os.listdir("/content/out_model"))
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
                      "--format=csv"], capture_output=True, text=True).stdout)

functions alive: True
out_model exists: True
['model.safetensors', 'tokenizer.json', 'config.json', 'generation_config.json', 'chat_template.jinja', 'tokenizer_config.json']
memory.used [MiB], utilization.gpu [%]
1089 MiB, 0 %



In [12]:
import json, os

cfg = json.load(open("/content/out_model/config.json"))
qc = cfg.get("quantization_config", {})
print("quantization_config:", json.dumps(qc, indent=2))
print("disk size GB:", dir_size_gb("/content/out_model"))
print("sha256:", folder_sha256("/content/out_model"))

# if the report file from Step 3 exists, print it
if os.path.exists("/content/report.json"):
    print(open("/content/report.json").read())
else:
    print("report.json not found")

quantization_config: {
  "_load_in_4bit": false,
  "_load_in_8bit": true,
  "bnb_4bit_compute_dtype": "float32",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "fp4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": false,
  "load_in_8bit": true,
  "quant_method": "bitsandbytes"
}
disk size GB: 0.643
sha256: d77e8101b56bb22c8795327b521cc5e0e26976aadc15fe86a44a3c50f81fc857
report.json not found


In [13]:
import warnings, logging, json, torch
warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(MODEL)

base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
base_ppl, base_gb = eval_perplexity(base, tok), base.get_memory_footprint() / 1e9
del base; free_gpu()

win = AutoModelForCausalLM.from_pretrained("/content/out_model", device_map="cuda")
win_ppl, win_gb = eval_perplexity(win, tok), win.get_memory_footprint() / 1e9
del win; free_gpu()

report = {
    "model_id": MODEL,
    "config": "int8",
    "fp16_size_gb": round(base_gb, 3),
    "final_size_gb": round(win_gb, 3),
    "compression_x": round(base_gb / win_gb, 2),
    "fp16_ppl": round(base_ppl, 3),
    "final_ppl": round(win_ppl, 3),
    "ppl_increase_pct": round((win_ppl - base_ppl) / base_ppl * 100, 2),
    "budget_pct": 10.0,
    "artifact_sha256": folder_sha256("/content/out_model"),
}
json.dump(report, open("/content/report.json", "w"), indent=2)
print(json.dumps(report, indent=2))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{
  "model_id": "Qwen/Qwen2.5-0.5B",
  "config": "int8",
  "fp16_size_gb": 0.988,
  "final_size_gb": 0.63,
  "compression_x": 1.57,
  "fp16_ppl": 17.261,
  "final_ppl": 17.415,
  "ppl_increase_pct": 0.89,
  "budget_pct": 10.0,
  "artifact_sha256": "d77e8101b56bb22c8795327b521cc5e0e26976aadc15fe86a44a3c50f81fc857"
}


In [14]:
print(open("/content/report.json").read())

{
  "model_id": "Qwen/Qwen2.5-0.5B",
  "config": "int8",
  "fp16_size_gb": 0.988,
  "final_size_gb": 0.63,
  "compression_x": 1.57,
  "fp16_ppl": 17.261,
  "final_ppl": 17.415,
  "ppl_increase_pct": 0.89,
  "budget_pct": 10.0,
  "artifact_sha256": "d77e8101b56bb22c8795327b521cc5e0e26976aadc15fe86a44a3c50f81fc857"
}


In [15]:
import json, os, time, shutil, torch, warnings
warnings.filterwarnings("ignore")
from transformers import AutoModelForCausalLM, AutoTokenizer

MEM = "/content/memory.json"
def mem_load(): return json.load(open(MEM)) if os.path.exists(MEM) else {}
def mem_save(m): json.dump(m, open(MEM, "w"), indent=2)

# seed memory with what we already measured
m = mem_load()
e = m.setdefault("Qwen/Qwen2.5-0.5B", {"configs": {}})
e.update({"fp16_ppl": 17.261, "fp16_gb": 0.988})
e["configs"]["nf4+double_quant"] = {"ppl_increase_pct": 12.92, "size_gb": 0.466}
e["configs"]["int8"] = {"ppl_increase_pct": 0.89, "size_gb": 0.63}
mem_save(m)

def quantize_smart(model_id, budget=10.0, bits="auto", out_dir="/content/out_model"):
    mem = mem_load()
    entry = mem.setdefault(model_id, {"configs": {}})
    tok = AutoTokenizer.from_pretrained(model_id)

    ladder = build_ladder()
    if bits in (4, 8):
        ladder = [(n, c) for n, c in ladder if n.startswith("nf4") == (bits == 4)]

    if "fp16_ppl" not in entry:   # measure baseline only once per model
        base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda")
        entry["fp16_gb"] = round(base.get_memory_footprint() / 1e9, 3)
        entry["fp16_ppl"] = eval_perplexity(base, tok)
        del base; free_gpu(); mem_save(mem)

    attempts, winner = [], None
    for name, cfg in ladder:
        known = entry["configs"].get(name)
        if known and known["ppl_increase_pct"] > budget:
            attempts.append({"config": name, **known, "source": "memory", "verdict": "FAIL"})
            print(f"[{name}] skipped, known +{known['ppl_increase_pct']}% (memory)")
            continue

        qm = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=cfg, device_map="cuda")
        if known:   # known to pass: no need to re-evaluate
            res, src = dict(known), "memory"
        else:
            q_ppl = eval_perplexity(qm, tok)
            res = {"ppl_increase_pct": round((q_ppl - entry["fp16_ppl"]) / entry["fp16_ppl"] * 100, 2),
                   "size_gb": round(qm.get_memory_footprint() / 1e9, 3)}
            entry["configs"][name] = res; mem_save(mem); src = "measured"
        ok = res["ppl_increase_pct"] <= budget
        attempts.append({"config": name, **res, "source": src, "verdict": "PASS" if ok else "FAIL"})
        print(f"[{name}] +{res['ppl_increase_pct']}% size={res['size_gb']} GB -> {'PASS' if ok else 'FAIL'} ({src})")
        if ok:
            if os.path.exists(out_dir): shutil.rmtree(out_dir)
            qm.save_pretrained(out_dir); tok.save_pretrained(out_dir)
            winner = name; del qm; free_gpu(); break
        del qm; free_gpu()

    return {
        "model_id": model_id, "budget_pct": budget, "bits_requested": bits,
        "fp16_size_gb": entry["fp16_gb"], "winner": winner,
        "winner_stats": entry["configs"].get(winner),
        "attempts": attempts,
        "artifact_dir": out_dir if winner else None,
        "artifact_sha256": folder_sha256(out_dir) if winner else None,
        "status": "SUCCESS" if winner else "NO_CONFIG_MET_BUDGET",
    }

# Test 1: strict budget, auto mode
r1 = quantize_smart("Qwen/Qwen2.5-0.5B", budget=10.0, bits="auto")
# Test 2: relaxed budget, smaller file allowed
r2 = quantize_smart("Qwen/Qwen2.5-0.5B", budget=15.0, bits="auto")

out = {"budget_10": r1, "budget_15": r2}
json.dump(out, open("/content/report2.json", "w"), indent=2)
print("\n=== SUMMARY ===")
for k, r in out.items():
    print(k, "->", r["status"], r["winner"], r["winner_stats"])

[nf4+double_quant] skipped, known +12.92% (memory)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors


[nf4] +12.95% size=0.451 GB -> FAIL (measured)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[int8] +0.89% size=0.63 GB -> PASS (memory)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[nf4+double_quant] +12.92% size=0.466 GB -> PASS (memory)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== SUMMARY ===
budget_10 -> SUCCESS int8 {'ppl_increase_pct': 0.89, 'size_gb': 0.63}
budget_15 -> SUCCESS nf4+double_quant {'ppl_increase_pct': 12.92, 'size_gb': 0.466}


In [16]:
import json
r = json.load(open("/content/report2.json"))
for k, v in r.items():
    print(k, "->", v["status"], v["winner"], v["winner_stats"])
    for a in v["attempts"]:
        print("   ", a["config"], a["ppl_increase_pct"], a["size_gb"], a["verdict"], a["source"])

budget_10 -> SUCCESS int8 {'ppl_increase_pct': 0.89, 'size_gb': 0.63}
    nf4+double_quant 12.92 0.466 FAIL memory
    nf4 12.95 0.451 FAIL measured
    int8 0.89 0.63 PASS memory
budget_15 -> SUCCESS nf4+double_quant {'ppl_increase_pct': 12.92, 'size_gb': 0.466}
    nf4+double_quant 12.92 0.466 PASS memory


In [17]:
!pip install -q fastapi uvicorn

import threading, time, json, requests, uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Quantize-as-a-tool")
gpu_lock = threading.Lock()

class Req(BaseModel):
    model_id: str
    accuracy_budget_pct: float = 10.0
    bits: str = "auto"          # "auto", "4" or "8"

@app.post("/quantize")
def quantize_endpoint(req: Req):
    insp = fetch_model_report(req.model_id)
    if not insp["safe_to_proceed"]:
        return {"status": "REFUSED", "flags": insp["flags"]}
    bits = int(req.bits) if req.bits in ("4", "8") else "auto"
    out_dir = f"/content/artifacts/{req.model_id.replace('/', '_')}_{req.accuracy_budget_pct}"
    with gpu_lock:
        return quantize_smart(req.model_id, req.accuracy_budget_pct, bits, out_dir)

@app.get("/health")
def health():
    return {"ok": True}

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

# act as a calling agent
r = requests.post("http://127.0.0.1:8000/quantize",
                  json={"model_id": "Qwen/Qwen2.5-0.5B", "accuracy_budget_pct": 10.0}, timeout=900)
data = r.json()
json.dump(data, open("/content/api_result.json", "w"), indent=2)
print(data["status"], data["winner"], data["winner_stats"], data["artifact_dir"])

[nf4+double_quant] skipped, known +12.92% (memory)
[nf4] skipped, known +12.95% (memory)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[int8] +0.89% size=0.63 GB -> PASS (memory)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SUCCESS int8 {'ppl_increase_pct': 0.89, 'size_gb': 0.63} /content/artifacts/Qwen_Qwen2.5-0.5B_10.0


In [18]:
import json
d = json.load(open("/content/api_result.json"))
print(d["status"], d["winner"], d["winner_stats"], d["artifact_dir"])
for a in d["attempts"]:
    print("  ", a["config"], a["ppl_increase_pct"], a["size_gb"], a["verdict"], a["source"])

SUCCESS int8 {'ppl_increase_pct': 0.89, 'size_gb': 0.63} /content/artifacts/Qwen_Qwen2.5-0.5B_10.0
   nf4+double_quant 12.92 0.466 FAIL memory
   nf4 12.95 0.451 FAIL memory
   int8 0.89 0.63 PASS memory


In [19]:
from IPython.display import HTML
HTML("<pre style='font-size:16px'>" + open("/content/api_result.json").read() + "</pre>")

In [20]:
import requests, json

r = requests.post("http://127.0.0.1:8000/quantize",
                  json={"model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                        "accuracy_budget_pct": 10.0}, timeout=1800)
d = r.json()
json.dump(d, open("/content/api_result_tinyllama.json", "w"), indent=2)

print(d["status"], d.get("winner"), d.get("winner_stats"), d.get("artifact_dir"))
for a in d.get("attempts", []):
    print("  ", a["config"], a["ppl_increase_pct"], a["size_gb"], a["verdict"], a["source"])

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

... (4 outputs left out)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[nf4+double_quant] +5.42% size=0.747 GB -> PASS (measured)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SUCCESS nf4+double_quant {'ppl_increase_pct': 5.42, 'size_gb': 0.747} /content/artifacts/TinyLlama_TinyLlama-1.1B-Chat-v1.0_10.0
   nf4+double_quant 5.42 0.747 PASS measured


In [21]:
!pip install -q pycloudflared

import json, time

# --- 1. size guard: patch the existing endpoint's logic ---
MAX_PARAMS_M = 3000   # refuse anything above 3B params on a T4

_orig_fetch = fetch_model_report
def fetch_model_report_guarded(model_id):
    rep = _orig_fetch(model_id)
    if rep["params_millions"] and rep["params_millions"] > MAX_PARAMS_M:
        rep["flags"].append(f"TOO_LARGE: {rep['params_millions']}M params > {MAX_PARAMS_M}M limit")
        rep["safe_to_proceed"] = False
    return rep

fetch_model_report = fetch_model_report_guarded   # the endpoint looks this name up at call time

# --- 2. public URL ---
from pycloudflared import try_cloudflare
tunnel = try_cloudflare(port=8000)
url = tunnel.tunnel
open("/content/public_url.txt", "w").write(url)
print("PUBLIC URL:", url)
print("Test in your browser:", url + "/health")

Download cloudflared...:   0%|          | 0/40346301 [00:00<?, ?it/s]

 * Running on https://wells-mic-calculations-accepting.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics
PUBLIC URL: https://wells-mic-calculations-accepting.trycloudflare.com
Test in your browser: https://wells-mic-calculations-accepting.trycloudflare.com/health


In [23]:
import requests, json

def fetch_model_report_safe(model_id):
    try:
        rep = _orig_fetch(model_id)           # the original fetcher from Cell 2
    except Exception as e:
        name = type(e).__name__
        flag = ("GATED: needs license acceptance and an HF token" if name == "GatedRepoError"
                else "NOT_FOUND: repo does not exist or is private" if name in ("RepositoryNotFoundError", "HfHubHTTPError")
                else f"FETCH_ERROR: {name}")
        return {"model_id": model_id, "params_millions": None,
                "flags": [flag], "safe_to_proceed": False}
    if rep["params_millions"] and rep["params_millions"] > MAX_PARAMS_M:
        rep["flags"].append(f"TOO_LARGE: {rep['params_millions']}M params > {MAX_PARAMS_M}M limit")
        rep["safe_to_proceed"] = False
    return rep

fetch_model_report = fetch_model_report_safe   # the endpoint looks this name up at call time

url = open("/content/public_url.txt").read().strip()
tests = ["meta-llama/Llama-2-13b-hf",   # gated
         "Qwen/Qwen2.5-7B",             # too large
         "does-not/exist-123"]          # missing
for m in tests:
    r = requests.post(url + "/quantize", json={"model_id": m, "accuracy_budget_pct": 10.0}, timeout=120)
    print(m, "->", r.status_code, r.text[:200])

meta-llama/Llama-2-13b-hf -> 200 {"status":"REFUSED","flags":["GATED: needs license acceptance and an HF token"]}


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

Qwen/Qwen2.5-7B -> 200 {"status":"REFUSED","flags":["TOO_LARGE: 7615.6M params > 3000M limit"]}
does-not/exist-123 -> 200 {"status":"REFUSED","flags":["NOT_FOUND: repo does not exist or is private"]}


In [25]:
import secrets, requests, json, time, threading, uvicorn
from starlette.responses import JSONResponse

# stop the old server (frees port 8000)
server.should_exit = True
time.sleep(4)

API_KEY = secrets.token_urlsafe(24)
open("/content/api_key.txt", "w").write(API_KEY)

class KeyGate:
    """Plain ASGI wrapper: /quantize needs the X-API-Key header, everything else is open."""
    def __init__(self, inner):
        self.inner = inner
    async def __call__(self, scope, receive, send):
        if scope["type"] == "http" and scope["path"] == "/quantize":
            supplied = dict(scope["headers"]).get(b"x-api-key", b"")
            if not secrets.compare_digest(supplied, API_KEY.encode()):
                resp = JSONResponse({"status": "UNAUTHORIZED"}, status_code=401)
                await resp(scope, receive, send)
                return
        await self.inner(scope, receive, send)

server = uvicorn.Server(uvicorn.Config(KeyGate(app), host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(4)

url = open("/content/public_url.txt").read().strip()
body = {"model_id": "Qwen/Qwen2.5-0.5B", "accuracy_budget_pct": 15.0}

r1 = requests.post(url + "/quantize", json=body, timeout=60)
print("no key   ->", r1.status_code, r1.text[:100])

r2 = requests.post(url + "/quantize", json=body, headers={"X-API-Key": API_KEY}, timeout=900)
print("with key ->", r2.status_code, r2.text[:300])

no key   -> 401 {"status":"UNAUTHORIZED"}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[nf4+double_quant] +12.92% size=0.466 GB -> PASS (memory)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

with key -> 200 {"model_id":"Qwen/Qwen2.5-0.5B","budget_pct":15.0,"bits_requested":"auto","fp16_size_gb":0.988,"winner":"nf4+double_quant","winner_stats":{"ppl_increase_pct":12.92,"size_gb":0.466},"attempts":[{"config":"nf4+double_quant","ppl_increase_pct":12.92,"size_gb":0.466,"source":"memory","verdict":"PASS"}],


In [26]:
%%writefile /content/quantize_service.py
import os, json, time, math, gc, shutil, hashlib, secrets, threading, warnings
warnings.filterwarnings("ignore")
import torch
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MEM = os.environ.get("MEM_PATH", "/content/drive/MyDrive/quantize_agent/memory.json")
MAX_PARAMS_M = 3000
api = HfApi()

# ---------------- helpers ----------------
def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

def dir_size_gb(path):
    return round(sum(os.path.getsize(os.path.join(r, f))
                     for r, _, fs in os.walk(path) for f in fs) / 1e9, 3)

def folder_sha256(path):
    h = hashlib.sha256()
    for f in sorted(os.listdir(path)):
        if f.endswith(".safetensors"):
            with open(os.path.join(path, f), "rb") as fh:
                for block in iter(lambda: fh.read(1 << 20), b""):
                    h.update(block)
    return h.hexdigest()

_WIKI = None
def eval_perplexity(model, tok, max_tokens=40000, seq_len=512):
    global _WIKI
    if _WIKI is None:
        from datasets import load_dataset
        d = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
        _WIKI = "\n\n".join(d["text"])
    ids = tok(_WIKI, return_tensors="pt").input_ids[:, :max_tokens]
    nll, count = 0.0, 0
    model.eval()
    with torch.no_grad():
        for i in range(0, ids.size(1) - 1, seq_len):
            chunk = ids[:, i:i + seq_len].to("cuda")
            if chunk.size(1) < 2:
                break
            out = model(chunk, labels=chunk)
            n = chunk.size(1) - 1
            nll += out.loss.item() * n
            count += n
    return math.exp(nll / count)

# ---------------- agent 1: fetcher ----------------
def _fetch_raw(model_id):
    info = api.model_info(model_id, files_metadata=True)
    files = {s.rfilename: (s.size or 0) for s in info.siblings}
    st = {n: s for n, s in files.items() if n.endswith(".safetensors")}
    pk = {n: s for n, s in files.items() if n.endswith((".bin", ".pt", ".pth", ".pkl"))}
    cfg = json.load(open(hf_hub_download(model_id, "config.json")))
    params = None
    if getattr(info, "safetensors", None) and info.safetensors.total:
        params = info.safetensors.total
    elif st:
        params = int(sum(st.values()) / 2)
    flags = []
    if cfg.get("auto_map"): flags.append("REMOTE_CODE: needs trust_remote_code")
    if pk and not st: flags.append("PICKLE_ONLY: unsafe weights")
    if getattr(info, "gated", False): flags.append("GATED: needs license and token")
    if cfg.get("quantization_config"): flags.append("ALREADY_QUANTIZED")
    return {
        "model_id": model_id,
        "architecture": (cfg.get("architectures") or ["?"])[0],
        "params_millions": round(params / 1e6, 1) if params else None,
        "flags": flags,
        "safe_to_proceed": not any(x.startswith(("REMOTE_CODE", "PICKLE_ONLY", "GATED")) for x in flags),
    }

def fetch_model_report(model_id):
    try:
        rep = _fetch_raw(model_id)
    except Exception as e:
        n = type(e).__name__
        flag = ("GATED: needs license acceptance and an HF token" if n == "GatedRepoError"
                else "NOT_FOUND: repo does not exist or is private" if n in ("RepositoryNotFoundError", "HfHubHTTPError")
                else f"FETCH_ERROR: {n}")
        return {"model_id": model_id, "params_millions": None, "flags": [flag], "safe_to_proceed": False}
    if rep["params_millions"] and rep["params_millions"] > MAX_PARAMS_M:
        rep["flags"].append(f"TOO_LARGE: {rep['params_millions']}M params > {MAX_PARAMS_M}M limit")
        rep["safe_to_proceed"] = False
    return rep

# ---------------- agent 2: quantizer (with memory) ----------------
def build_ladder():
    return [
        ("nf4+double_quant", BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)),
        ("nf4", BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=False, bnb_4bit_compute_dtype=torch.float16)),
        ("int8", BitsAndBytesConfig(load_in_8bit=True)),
    ]

def mem_load():
    return json.load(open(MEM)) if os.path.exists(MEM) else {}

def mem_save(m):
    os.makedirs(os.path.dirname(MEM), exist_ok=True)
    json.dump(m, open(MEM, "w"), indent=2)

def quantize_smart(model_id, budget=10.0, bits="auto", out_dir="/content/out_model"):
    mem = mem_load()
    entry = mem.setdefault(model_id, {"configs": {}})
    tok = AutoTokenizer.from_pretrained(model_id)
    ladder = build_ladder()
    if bits in (4, 8):
        ladder = [(n, c) for n, c in ladder if n.startswith("nf4") == (bits == 4)]
    if "fp16_ppl" not in entry:
        base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda")
        entry["fp16_gb"] = round(base.get_memory_footprint() / 1e9, 3)
        entry["fp16_ppl"] = eval_perplexity(base, tok)
        del base; free_gpu(); mem_save(mem)
    attempts, winner = [], None
    for name, cfg in ladder:
        known = entry["configs"].get(name)
        if known and known["ppl_increase_pct"] > budget:
            attempts.append({"config": name, **known, "source": "memory", "verdict": "FAIL"})
            continue
        qm = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=cfg, device_map="cuda")
        if known:
            res, src = dict(known), "memory"
        else:
            q_ppl = eval_perplexity(qm, tok)
            res = {"ppl_increase_pct": round((q_ppl - entry["fp16_ppl"]) / entry["fp16_ppl"] * 100, 2),
                   "size_gb": round(qm.get_memory_footprint() / 1e9, 3)}
            entry["configs"][name] = res; mem_save(mem); src = "measured"
        ok = res["ppl_increase_pct"] <= budget
        attempts.append({"config": name, **res, "source": src, "verdict": "PASS" if ok else "FAIL"})
        print(f"[{name}] +{res['ppl_increase_pct']}% size={res['size_gb']} GB -> {'PASS' if ok else 'FAIL'} ({src})")
        if ok:
            if os.path.exists(out_dir): shutil.rmtree(out_dir)
            qm.save_pretrained(out_dir); tok.save_pretrained(out_dir)
            winner = name; del qm; free_gpu(); break
        del qm; free_gpu()
    return {
        "model_id": model_id, "budget_pct": budget, "bits_requested": bits,
        "fp16_size_gb": entry["fp16_gb"], "winner": winner,
        "winner_stats": entry["configs"].get(winner), "attempts": attempts,
        "artifact_dir": out_dir if winner else None,
        "artifact_sha256": folder_sha256(out_dir) if winner else None,
        "status": "SUCCESS" if winner else "NO_CONFIG_MET_BUDGET",
    }

# ---------------- agent 3: validator ----------------
def verify_artifact(artifact_dir, expected_sha256):
    ok = os.path.exists(artifact_dir) and folder_sha256(artifact_dir) == expected_sha256
    return {"hash_matches": ok, "size_gb": dir_size_gb(artifact_dir) if ok else None}

# ---------------- service ----------------
from fastapi import FastAPI
from pydantic import BaseModel
from starlette.responses import JSONResponse

app = FastAPI(title="Quantize-as-a-tool")
gpu_lock = threading.Lock()

class Req(BaseModel):
    model_id: str
    accuracy_budget_pct: float = 10.0
    bits: str = "auto"

@app.post("/quantize")
def quantize_endpoint(req: Req):
    insp = fetch_model_report(req.model_id)
    if not insp["safe_to_proceed"]:
        return {"status": "REFUSED", "flags": insp["flags"]}
    bits = int(req.bits) if req.bits in ("4", "8") else "auto"
    out_dir = f"/content/artifacts/{req.model_id.replace('/', '_')}_{req.accuracy_budget_pct}"
    with gpu_lock:
        return quantize_smart(req.model_id, req.accuracy_budget_pct, bits, out_dir)

@app.get("/health")
def health():
    return {"ok": True}

class KeyGate:
    def __init__(self, inner, key):
        self.inner, self.key = inner, key.encode()
    async def __call__(self, scope, receive, send):
        if scope["type"] == "http" and scope["path"] == "/quantize":
            supplied = dict(scope["headers"]).get(b"x-api-key", b"")
            if not secrets.compare_digest(supplied, self.key):
                await JSONResponse({"status": "UNAUTHORIZED"}, status_code=401)(scope, receive, send)
                return
        await self.inner(scope, receive, send)

def start_service(port=8000, tunnel=True):
    import uvicorn
    key = secrets.token_urlsafe(24)
    open("/content/api_key.txt", "w").write(key)
    server = uvicorn.Server(uvicorn.Config(KeyGate(app, key), host="127.0.0.1", port=port, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    time.sleep(4)
    url = None
    if tunnel:
        from pycloudflared import try_cloudflare
        url = try_cloudflare(port=port).tunnel
        open("/content/public_url.txt", "w").write(url)
    return server, key, url

Writing /content/quantize_service.py


In [27]:
import os, shutil
from google.colab import files

repo = "/content/quantize-as-a-tool"
shutil.rmtree(repo, ignore_errors=True)
os.makedirs(repo + "/results")

# 1. service code, with memory stored next to the script instead of Drive
src = open("/content/quantize_service.py").read()
src = src.replace('"/content/drive/MyDrive/quantize_agent/memory.json"',
                  'os.path.join(os.path.dirname(os.path.abspath(__file__)), "memory.json")')
open(repo + "/quantize_service.py", "w").write(src)

# 2. measured results
for f in ["memory.json"]:
    if os.path.exists("/content/" + f):
        shutil.copy("/content/" + f, repo + "/" + f)
for f in ["report.json", "report2.json", "api_result.json", "api_result_tinyllama.json"]:
    if os.path.exists("/content/" + f):
        shutil.copy("/content/" + f, repo + "/results/" + f)

# 3. requirements, gitignore, README
open(repo + "/requirements.txt", "w").write(
    "transformers\naccelerate\nbitsandbytes\nhuggingface_hub\nsafetensors\n"
    "datasets\nfastapi\nuvicorn\npycloudflared\n")

open(repo + "/.gitignore", "w").write(
    "api_key.txt\npublic_url.txt\nartifacts/\nout_model/\n__pycache__/\n*.safetensors\n.env\n")

open(repo + "/README.md", "w").write("""# Quantize-as-a-Tool

Model compression that AI agents can call over HTTP, with an accuracy budget and a verification report.

An agent sends `{model_id, accuracy_budget_pct}` and gets back the smallest 4-bit or 8-bit
version of a Hugging Face model that stays within its perplexity budget.

## How it works
1. **Fetcher** reads metadata only (no weights) and refuses gated, pickle-only, remote-code or oversized (>3B) models.
2. **Quantizer** tries NF4+double-quant, NF4, then int8, and stops at the first one that meets the budget.
3. **Validator** compares WikiText-2 perplexity with the fp16 baseline and records a SHA-256 of the artifact.

A per-model memory (`memory.json`) makes repeat requests instant.

## Measured results (Tesla T4)
| Model | Format chosen | Size vs fp16 | Perplexity change | Budget |
|---|---|---|---|---|
| Qwen2.5-0.5B | int8 | 0.63 GB vs 0.99 GB | +0.89% | 10% |
| Qwen2.5-0.5B | NF4 4-bit | 0.47 GB | +12.92% | 15% |
| TinyLlama-1.1B | NF4 4-bit | 0.75 GB vs 2.2 GB | +5.42% | 10% |

The right format depends on the model: 4-bit fails a 10% budget on Qwen 0.5B but passes on TinyLlama 1.1B.

## Run it (Google Colab, T4 GPU)
```python
!pip install -q -r requirements.txt
%run quantize_service.py
server, key, url = start_service()
print(url)   # API key is saved in /content/api_key.txt
```

## Call it
```bash
curl -X POST $URL/quantize \\
  -H "X-API-Key: $KEY" -H "Content-Type: application/json" \\
  -d '{"model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "accuracy_budget_pct": 10}'
```

## Limits
Perplexity is a proxy for quality (task benchmarks are next). Tested on two small models. Deterministic Python agents, no LLM in the loop.
Roadmap: MCP wrapper, more architectures, GPTQ/AWQ/GGUF, signed reports.
""")

# 4. zip and download
shutil.make_archive("/content/quantize-as-a-tool", "zip", repo)
print(sorted(os.listdir(repo)))
files.download("/content/quantize-as-a-tool.zip")

['.gitignore', 'README.md', 'memory.json', 'quantize_service.py', 'requirements.txt', 'results']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>